Questo notebook fa parte di **Pensare in Python**, traduzione italiana non ufficiale di [*Think Python*, 3ª edizione](https://allendowney.github.io/ThinkPython/) di Allen B. Downey. Puoi leggere tutto il libro tradotto su <https://pensare-in-python.github.io/3a-edizione/>.

Le versioni stampata ed ebook dell'originale si possono acquistare da [Bookshop.org](https://bookshop.org/a/98697/9781098155438) e [Amazon](https://www.amazon.com/_/dp/1098155432?smid=ATVPDKIKX0DER&_encoding=UTF8&tag=greenteapre01-20&linkCode=ur2&linkId=e2a529f94920295d27ec8a06e757dc7c&camp=1789&creative=9325) (link di affiliazione dell'autore).

Se trovi utile questo libro, valuta di [sostenere i materiali didattici liberi e aperti](https://greenteapress.com/support/?book=tp3) di Green Tea Press.

In [ ]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print("Downloaded " + str(local))
    return filename

download('https://github.com/AllenDowney/ThinkPython/raw/v3/thinkpython.py');
download('https://github.com/AllenDowney/ThinkPython/raw/v3/diagram.py');

import thinkpython

Fonte: le foto sono scaricate da [Lorem Picsum](https://picsum.photos/), un servizio che fornisce immagini segnaposto.
Il nome richiama "lorem ipsum", il nome con cui si indica il testo segnaposto.

In [ ]:
# This cell downloads an archive file that contains the the files we'll
# use for the examples in this chapter.

download('https://github.com/AllenDowney/ThinkPython/raw/v3/photos.zip');

In [ ]:
# WARNING: This cell removes the photos/ directory if it already exists.
# Any files already in the photos/ directory will be deleted.

# !rm -rf photos/

In [ ]:
!unzip -o photos.zip

# File e database

[Esegui questo notebook su Colab](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap13.ipynb)

La maggior parte dei programmi visti finora è **effimera** (*ephemeral*), nel senso che viene eseguita per poco tempo e produce un output, ma quando termina i suoi dati scompaiono.
Ogni volta che esegui un programma effimero, riparte da zero.

Altri programmi sono **persistenti** (*persistent*): vengono eseguiti a lungo (o sempre); conservano almeno una parte dei loro dati in una memoria permanente; e se si arrestano e ripartono, riprendono da dove avevano lasciato.

Un modo semplice per i programmi di conservare i propri dati è leggere e scrivere file di testo.
Un'alternativa più versatile è memorizzare i dati in un database.
I database sono file specializzati che si possono leggere e scrivere in modo più efficiente dei file di testo, e offrono funzionalità aggiuntive.

In questo capitolo scriveremo programmi che leggono e scrivono file di testo e database, e come esercizio scriverai un programma che cerca i duplicati in una raccolta di foto.
Ma prima di poter lavorare con un file, devi trovarlo, quindi inizieremo con nomi di file, percorsi e directory.

## Nomi di file e percorsi

I file sono organizzati in **directory**, chiamate anche "cartelle".
Ogni programma in esecuzione ha una **directory di lavoro corrente** (*current working directory*), che è la directory predefinita per la maggior parte delle operazioni.
Per esempio, quando apri un file, Python lo cerca nella directory di lavoro corrente.

Il modulo `os` fornisce funzioni per lavorare con file e directory ("os" sta per "operating system", sistema operativo).
Fornisce una funzione chiamata `getcwd` che ottiene il nome della directory di lavoro corrente.

In [ ]:
# this cell replaces `os.cwd` with a function that returns a fake path

import os

def getcwd():
    return "/home/dinsdale"

os.getcwd = getcwd

In [ ]:
import os

os.getcwd()

Il risultato in questo esempio è la directory home di un utente chiamato `dinsdale`.
Una stringa come `'/home/dinsdale'` che identifica un file o una directory si chiama **percorso** (*path*).

Anche un semplice nome di file come `'memo.txt'` è considerato un percorso, ma è un **percorso relativo** (*relative path*) perché specifica un nome di file rispetto alla directory corrente.
In questo esempio la directory corrente è `/home/dinsdale`, quindi `'memo.txt'` equivale al percorso completo `'/home/dinsdale/memo.txt'`.

Un percorso che inizia con `/` non dipende dalla directory corrente – si chiama **percorso assoluto** (*absolute path*).
Per trovare il percorso assoluto di un file, puoi usare `abspath`.

In [ ]:
os.path.abspath('memo.txt')

Il modulo `os` fornisce altre funzioni per lavorare con nomi di file e percorsi.
`listdir` restituisce una lista del contenuto della directory data, compresi file e altre directory.
Ecco un esempio che elenca il contenuto di una directory chiamata `photos`.

In [ ]:
os.listdir('photos')

Questa directory contiene un file di testo chiamato `notes.txt` e tre directory.
Le directory contengono file di immagine in formato JPEG.

In [ ]:
os.listdir('photos/jan-2023')

Per verificare se un file o una directory esiste, possiamo usare `os.path.exists`.

In [ ]:
os.path.exists('photos')

In [ ]:
os.path.exists('photos/apr-2023')

Per verificare se un percorso si riferisce a un file o a una directory, possiamo usare `isdir`, che restituisce `True` se un percorso si riferisce a una directory.

In [ ]:
os.path.isdir('photos')

E `isfile`, che restituisce `True` se un percorso si riferisce a un file.

In [ ]:
os.path.isfile('photos/notes.txt')

Una difficoltà del lavorare con i percorsi è che hanno un aspetto diverso su sistemi operativi diversi.
Su macOS e sui sistemi UNIX come Linux, i nomi di directory e file in un percorso sono separati da una barra, `/`.
Windows usa una barra rovesciata (*backslash*), `\`.
Quindi, se esegui questi esempi su Windows, vedrai barre rovesciate nei percorsi, e dovrai sostituire le barre negli esempi.

Oppure, per scrivere codice che funzioni su entrambi i sistemi, puoi usare `os.path.join`, che unisce nomi di directory e di file in un percorso usando una barra o una barra rovesciata, a seconda del sistema operativo che stai usando.

In [ ]:
os.path.join('photos', 'jan-2023', 'photo1.jpg')

Più avanti in questo capitolo useremo queste funzioni per esplorare un insieme di directory e trovare tutti i file di immagine.

## f-string

Un modo per i programmi di memorizzare dati è scriverli in un file di testo.
Per esempio, supponiamo che tu sia un osservatore di cammelli e che tu voglia registrare il numero di cammelli che hai visto durante un periodo di osservazione.
E supponiamo che in un anno e mezzo tu abbia avvistato `23` cammelli.
I dati nel tuo quaderno di avvistamenti potrebbero essere questi.

In [ ]:
num_years = 1.5
num_camels = 23

Per scrivere questi dati in un file, puoi usare il metodo `write`, che abbiamo visto nel capitolo 8.
L'argomento di `write` deve essere una stringa, quindi se vogliamo mettere altri valori in un file, dobbiamo convertirli in stringhe.
Il modo più semplice per farlo è con la funzione predefinita `str`.

Ecco come appare:

In [ ]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(str(num_years))
writer.write(str(num_camels))
writer.close()

Funziona, ma `write` non aggiunge uno spazio o un a capo a meno che tu non li includa esplicitamente.
Se rileggiamo il file, vediamo che i due numeri sono attaccati.

In [ ]:
open('camel-spotting-book.txt').read()

Come minimo, dovremmo aggiungere uno spazio tra i numeri.
E già che ci siamo, aggiungiamo un testo esplicativo.

Per scrivere una combinazione di stringhe e altri valori, possiamo usare una **f-string**, cioè una stringa che ha la lettera `f` prima dell'apice di apertura e contiene una o più espressioni Python tra parentesi graffe.
La f-string seguente contiene un'espressione, che è un nome di variabile.

In [ ]:
f'I have spotted {num_camels} camels'

Il risultato è una stringa in cui l'espressione è stata valutata e sostituita con il risultato.
Le espressioni possono essere più di una.

In [ ]:
f'In {num_years} years I have spotted {num_camels} camels'

E le espressioni possono contenere operatori e chiamate di funzione.

In [ ]:
line = f'In {round(num_years * 12)} months I have spotted {num_camels} camels'
line

Quindi potremmo scrivere i dati in un file di testo così.

In [ ]:
writer = open('camel-spotting-book.txt', 'w')
writer.write(f'Years of observation: {num_years}\n')
writer.write(f'Camels spotted: {num_camels}\n')
writer.close()

Entrambe le f-string terminano con la sequenza `\n`, che aggiunge un carattere di a capo.

Possiamo rileggere il file così:

In [ ]:
data = open('camel-spotting-book.txt').read()
print(data)

In una f-string, un'espressione tra parentesi graffe viene convertita in stringa, quindi puoi includere liste, dizionari e altri tipi.

In [ ]:
t = [1, 2, 3]
d = {'one': 1}
f'Here is a list {t} and a dictionary {d}'

Se una f-string contiene un'espressione non valida, il risultato è un errore.

In [ ]:
%%expect TypeError

f'This is not a valid expression {t + 2}'

## YAML

Uno dei motivi per cui i programmi leggono e scrivono file è memorizzare **dati di configurazione** (*configuration data*), cioè informazioni che specificano cosa deve fare il programma e come.

Per esempio, in un programma che cerca foto duplicate, potremmo avere un dizionario chiamato `config` che contiene il nome della directory in cui cercare, il nome di un'altra directory in cui memorizzare i risultati e una lista di estensioni di file da usare per identificare i file di immagine.

Ecco come potrebbe apparire:

In [ ]:
config = {
    'photo_dir': 'photos',
    'data_dir': 'photo_info',
    'extensions': ['jpg', 'jpeg'],
}

Per scrivere questi dati in un file di testo, potremmo usare le f-string, come nella sezione precedente. Ma è più facile usare un modulo chiamato `yaml`, pensato proprio per questo genere di cose.

Il modulo `yaml` fornisce funzioni per lavorare con i file YAML, che sono file di testo formattati in modo da essere facili da leggere e scrivere per le persone *e* per i programmi.

Ecco un esempio che usa la funzione `dump` per scrivere il dizionario `config` in un file YAML.

In [ ]:
# this cell installs the pyyaml package, which provides the yaml module

try:
    import yaml
except ImportError:
    !pip install pyyaml

In [ ]:
import yaml

config_filename = 'config.yaml'
writer = open(config_filename, 'w')
yaml.dump(config, writer)
writer.close()

Se rileggiamo il contenuto del file, possiamo vedere com'è fatto il formato YAML.

In [ ]:
readback = open(config_filename).read()
print(readback)

Ora possiamo usare `safe_load` per rileggere il file YAML.

In [ ]:
reader = open(config_filename)
config_readback = yaml.safe_load(reader)
config_readback

Il risultato è un nuovo dizionario che contiene le stesse informazioni dell'originale, ma non è lo stesso dizionario.

In [ ]:
config is config_readback

Convertire un oggetto come un dizionario in una stringa si chiama **serializzazione** (*serialization*).
Riconvertire la stringa in un oggetto si chiama **deserializzazione** (*deserialization*).
Se serializzi e poi deserializzi un oggetto, il risultato dovrebbe essere equivalente all'originale.

## Shelve

Finora abbiamo letto e scritto file di testo – ora consideriamo i database.
Un **database** è un file organizzato per memorizzare dati.
Alcuni database sono organizzati come una tabella con righe e colonne di informazioni.
Altri sono organizzati come un dizionario che fa corrispondere chiavi a valori; a volte si chiamano **archivi chiave-valore** (*key-value stores*).

Il modulo `shelve` fornisce funzioni per creare e aggiornare un archivio chiave-valore chiamato "shelf" (scaffale).
Come esempio, creeremo uno shelf che contiene le didascalie delle figure nella directory `photos`.
Useremo il dizionario `config` per ottenere il nome della directory in cui mettere lo shelf.

In [ ]:
config['data_dir']

Possiamo usare `os.makedirs` per creare questa directory, se non esiste già.

In [ ]:
os.makedirs(config['data_dir'], exist_ok=True)

E `os.path.join` per creare un percorso che include il nome della directory e il nome del file dello shelf, `captions`.

In [ ]:
db_file = os.path.join(config['data_dir'], 'captions')
db_file

Ora possiamo usare `shelve.open` per aprire il file dello shelf.
L'argomento `c` indica che il file va creato se necessario.

In [ ]:
import shelve

db = shelve.open(db_file, 'c')
db

Se ottieni un errore come `db type could not be determined` (impossibile determinare il tipo di db), la causa più probabile è che esista già un file con lo stesso nome che non è un database shelve valido (per esempio, potrebbe essere danneggiato o creato da qualcos'altro).

In quel caso, la soluzione più semplice è eliminare il file esistente ed eseguire di nuovo il codice, così che `shelve.open` possa creare un nuovo database.

Il valore di ritorno è ufficialmente un oggetto `DbfilenameShelf`, chiamato più informalmente oggetto shelf.

L'oggetto shelf si comporta per molti versi come un dizionario.
Per esempio, possiamo usare l'operatore parentesi quadre per aggiungere un elemento, cioè una corrispondenza da una chiave a un valore.

In [ ]:
key = 'jan-2023/photo1.jpg' 
db[key] = 'Cat nose'

In questo esempio, la chiave è il percorso di un file di immagine e il valore è una stringa che descrive l'immagine.

Usiamo l'operatore parentesi quadre anche per cercare una chiave e ottenere il valore corrispondente.

In [ ]:
value = db[key]
value

Se fai un'altra assegnazione a una chiave esistente, `shelve` sostituisce il vecchio valore.

In [ ]:
db[key] = 'Close up view of a cat nose'
db[key]

Alcuni metodi dei dizionari, come `keys`, `values` e `items`, funzionano anche con gli oggetti shelf.

In [ ]:
list(db.keys())

In [ ]:
list(db.values())

Possiamo usare l'operatore `in` per verificare se una chiave compare nello shelf.

In [ ]:
key in db

E possiamo usare un'istruzione `for` per scorrere le chiavi.

In [ ]:
for key in db:
    print(key, ':', db[key])

Come con gli altri file, dovresti chiudere il database quando hai finito.

In [ ]:
db.close()

Ora, se elenchiamo il contenuto della directory dei dati, vediamo due file.

In [ ]:
# When you open a shelve file, a backup file is created that has the suffix `.bak`.
# If you run this notebook more than once, you might see that file left behind.
# This cell removes it so the output shown in the book is correct.

!rm -f photo_info/captions.bak

In [ ]:
os.listdir(config['data_dir'])

`captions.dat` contiene i dati che abbiamo appena memorizzato.
`captions.dir` contiene informazioni sull'organizzazione del database che rendono più efficiente l'accesso.
Il suffisso `dir` sta per "directory", ma non ha nulla a che fare con le directory contenenti file con cui abbiamo lavorato.

## Memorizzare strutture dati

Nell'esempio precedente, le chiavi e i valori dello shelf sono stringhe.
Ma possiamo usare uno shelf anche per contenere strutture dati come liste e dizionari.

Come esempio, riprendiamo l'esempio degli anagrammi da un esercizio del [capitolo 11](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap11.ipynb).
Ricorda che avevamo creato un dizionario che fa corrispondere a una stringa di lettere ordinate la
lista delle parole che si possono comporre con quelle lettere.
Per esempio, la chiave `'opst'` corrisponde alla lista `['opts', 'post', 'pots', 'spot', 'stop', 'tops']`.

Useremo la funzione seguente per ordinare le lettere di una parola.

In [ ]:
def sort_word(word):
    return ''.join(sorted(word))

Ed ecco un esempio.

In [ ]:
word = 'pots'
key = sort_word(word)
key

Ora apriamo uno shelf chiamato `anagram_map`.
L'argomento `'n'` significa che dobbiamo sempre creare un nuovo shelf vuoto, anche se ne esiste già uno.

In [ ]:
db = shelve.open('anagram_map', 'n')

Ora possiamo aggiungere un elemento allo shelf così.

In [ ]:
db[key] = [word]
db[key]

In questo elemento, la chiave è una stringa e il valore è una lista di stringhe.

Supponiamo ora di trovare un'altra parola che contiene le stesse lettere, come `tops`.

In [ ]:
word = 'tops'
key = sort_word(word)
key

La chiave è la stessa dell'esempio precedente, quindi vogliamo aggiungere una seconda parola alla stessa lista di stringhe.
Ecco come lo faremmo se `db` fosse un dizionario.

In [ ]:
db[key].append(word)          # INCORRECT

Ma se eseguiamo questo codice e poi cerchiamo la chiave nello shelf, sembra che non sia stato aggiornato.

In [ ]:
db[key]

Il problema è questo: quando cerchiamo la chiave, otteniamo una lista di stringhe, ma se modifichiamo la lista di stringhe, ciò non ha effetto sullo shelf.
Se vogliamo aggiornare lo shelf, dobbiamo leggere il vecchio valore, aggiornarlo e poi riscrivere il nuovo valore nello shelf.

In [ ]:
anagram_list = db[key]
anagram_list.append(word)
db[key] = anagram_list

Ora il valore nello shelf è aggiornato.

In [ ]:
db[key]

Come esercizio, puoi completare questo esempio leggendo l'elenco di parole e memorizzando tutti gli anagrammi in uno shelf.

In [ ]:
db.close()

## Verificare file equivalenti

Torniamo ora all'obiettivo di questo capitolo: cercare file diversi che contengono gli stessi dati.
Un modo per verificarlo è leggere il contenuto di entrambi i file e confrontarlo.

Se i file contengono immagini, dobbiamo aprirli con la modalità `'rb'`, dove `'r'` significa che vogliamo leggere il contenuto e `'b'` indica la **modalità binaria** (*binary mode*).
In modalità binaria, il contenuto non viene interpretato come testo – viene trattato come una sequenza di byte.

Ecco un esempio che apre e legge un file di immagine.

In [ ]:
path1 = 'photos/jan-2023/photo1.jpg'
data1 = open(path1, 'rb').read()
type(data1)

Il risultato di `read` è un oggetto `bytes` – come suggerisce il nome, contiene una sequenza di byte.

In generale, il contenuto di un file di immagine non è leggibile per le persone.
Ma se leggiamo il contenuto di un secondo file, possiamo usare l'operatore `==` per confrontarli.

In [ ]:
path2 = 'photos/jan-2023/photo2.jpg'
data2 = open(path2, 'rb').read()
data1 == data2

Questi due file non sono equivalenti.

Incapsuliamo in una funzione quello che abbiamo fatto finora.

In [ ]:
def same_contents(path1, path2):
    data1 = open(path1, 'rb').read()
    data2 = open(path2, 'rb').read()
    return data1 == data2

Se abbiamo solo due file, questa funzione è una buona opzione.
Ma supponiamo di avere un gran numero di file e di voler sapere se due di essi contengono gli stessi dati.
Sarebbe inefficiente confrontare ogni coppia di file.

Un'alternativa è usare una **funzione hash** (*hash function*), che prende il contenuto di un file e calcola un **digest** (impronta), che di solito è un intero grande.
Se due file contengono gli stessi dati, avranno lo stesso digest.
Se due file sono diversi, avranno *quasi sempre* digest diversi.

Il modulo `hashlib` fornisce diverse funzioni hash – quella che useremo si chiama `md5`.
Inizieremo usando `hashlib.md5` per creare un oggetto `HASH`.

In [ ]:
import hashlib

md5_hash = hashlib.md5()
type(md5_hash)

L'oggetto `HASH` fornisce un metodo `update` che riceve come argomento il contenuto del file.

In [ ]:
md5_hash.update(data1)

Ora possiamo usare `hexdigest` per ottenere il digest come stringa di cifre esadecimali che rappresentano un intero in base 16.

In [ ]:
digest = md5_hash.hexdigest()
digest

La funzione seguente incapsula questi passi.

In [ ]:
def md5_digest(filename):
    data = open(filename, 'rb').read()
    md5_hash = hashlib.md5()
    md5_hash.update(data)
    digest = md5_hash.hexdigest()
    return digest

Se calcoliamo l'hash del contenuto di un file diverso, possiamo confermare di ottenere un digest diverso.

In [ ]:
filename2 = 'photos/feb-2023/photo2.jpg'
md5_digest(filename2)

Ora abbiamo quasi tutto ciò che serve per trovare file equivalenti.
L'ultimo passo è esplorare una directory e trovare tutti i file di immagine.

## Esplorare le directory

La funzione seguente riceve come argomento la directory in cui vogliamo cercare.
Usa `listdir` per scorrere il contenuto della directory.
Quando trova un file, ne stampa il percorso completo.
Quando trova una directory, chiama sé stessa ricorsivamente per esplorare la sottodirectory.

In [ ]:
def walk(dirname):
    for name in os.listdir(dirname):
        path = os.path.join(dirname, name)

        if os.path.isfile(path):
            print(path)
        elif os.path.isdir(path):
            walk(path)

Possiamo usarla così:

In [ ]:
walk('photos')

L'ordine dei risultati dipende da dettagli del sistema operativo.

## Debugging

Quando leggi e scrivi file, potresti avere problemi con gli spazi.
Questi errori possono essere difficili da correggere perché i caratteri di spaziatura normalmente sono invisibili.
Per esempio, ecco una stringa che contiene spazi, una tabulazione rappresentata dalla sequenza `\t` e un a capo rappresentato dalla sequenza `\n`.
Quando la stampiamo, non vediamo i caratteri di spaziatura.

In [ ]:
s = '1 2\t 3\n 4'
print(s)

La funzione predefinita `repr` può aiutare. Riceve un oggetto qualsiasi come argomento e restituisce una rappresentazione dell'oggetto sotto forma di stringa.
Per le stringhe, rappresenta i caratteri di spaziatura con sequenze con la barra rovesciata.

In [ ]:
print(repr(s))

Questo può essere utile per il debugging.

Un altro problema che potresti incontrare è che sistemi diversi usano caratteri diversi per indicare la fine di una riga. Alcuni sistemi usano un a capo, rappresentato da `\n`. Altri usano un carattere di ritorno a inizio riga, rappresentato da `\r`.
Alcuni li usano entrambi. Se sposti file tra sistemi diversi, queste
incongruenze possono causare problemi.

Le maiuscole e minuscole nei nomi di file sono un altro problema che potresti incontrare se lavori con sistemi operativi diversi.
In macOS e UNIX, i nomi di file possono contenere lettere minuscole e maiuscole, cifre e la maggior parte dei simboli.
Ma molte applicazioni Windows ignorano la differenza tra minuscole e maiuscole, e diversi simboli ammessi in macOS e UNIX non sono ammessi in Windows.

## Glossario

**effimero** (*ephemeral*):
Un programma effimero di solito viene eseguito per poco tempo e, quando termina, i suoi dati vanno persi.

**persistente** (*persistent*):
Un programma persistente viene eseguito indefinitamente e conserva almeno una parte dei suoi dati in una memoria permanente.

**directory**:
Una raccolta di file e di altre directory.

**directory di lavoro corrente** (*current working directory*):
La directory predefinita usata da un programma, a meno che non ne venga specificata un'altra.

**percorso** (*path*):
Una stringa che specifica una sequenza di directory, che spesso porta a un file.

**percorso relativo** (*relative path*):
Un percorso che parte dalla directory di lavoro corrente, o da un'altra directory specificata.

**percorso assoluto** (*absolute path*):
Un percorso che non dipende dalla directory corrente.

**f-string**:
Una stringa che ha la lettera `f` prima dell'apice di apertura e contiene una o più espressioni tra parentesi graffe.

**dati di configurazione** (*configuration data*):
Dati, spesso memorizzati in un file, che specificano cosa deve fare un programma e come.

**serializzazione** (*serialization*):
La conversione di un oggetto in una stringa.

**deserializzazione** (*deserialization*):
La conversione di una stringa in un oggetto.

**database**:
Un file il cui contenuto è organizzato per eseguire certe operazioni in modo efficiente.

**archivio chiave-valore** (*key-value store*):
Un database il cui contenuto è organizzato come un dizionario, con chiavi che corrispondono a valori.

**modalità binaria** (*binary mode*):
Un modo di aprire un file in cui il contenuto viene interpretato come sequenza di byte invece che come sequenza di caratteri.

**funzione hash** (*hash function*):
Una funzione che riceve un oggetto e calcola un intero, chiamato a volte digest.

**digest**:
Il risultato di una funzione hash, soprattutto quando viene usato per verificare se due oggetti sono uguali.

## Esercizi

In [ ]:
# This cell tells Jupyter to provide detailed debugging information
# when a runtime error occurs. Run it before working on the exercises.

%xmode Verbose

### Chiedi a un assistente virtuale

In questo capitolo sono emersi diversi argomenti che non ho spiegato nel dettaglio.
Ecco alcune domande che puoi fare a un assistente virtuale per avere più informazioni.

* "Quali sono le differenze tra programmi effimeri e persistenti?"

* "Quali sono alcuni esempi di programmi persistenti?"

* "Qual è la differenza tra un percorso relativo e un percorso assoluto?"

* "Perché il modulo `yaml` ha funzioni chiamate `load` e `safe_load`?"

* "Quando scrivo uno shelf Python, cosa sono i file con suffisso `dat` e `dir`?"

* "Oltre agli archivi chiave-valore, quali altri tipi di database esistono?"

* "Quando leggo un file, qual è la differenza tra modalità binaria e modalità testo?"

* "Quali sono le differenze tra un oggetto bytes e una stringa?"

* "Che cos'è una funzione hash?"

* "Che cos'è un digest MD5?"

Come sempre, se resti bloccato su uno degli esercizi seguenti, valuta di chiedere aiuto a un assistente. Insieme alla domanda, potresti voler incollare le funzioni pertinenti di questo capitolo.

### Esercizio

Scrivi una funzione chiamata `replace_all` che riceve come argomenti una stringa schema, una stringa di sostituzione e due nomi di file.
Deve leggere il primo file e scriverne il contenuto nel secondo file (creandolo se necessario).
Se la stringa schema compare in qualche punto del contenuto, deve essere sostituita con la stringa di sostituzione.

Ecco uno schema della funzione per aiutarti a iniziare.

In [ ]:
def replace_all(old, new, source_path, dest_path):
    # read the contents of the source file
    reader = open(source_path)

    # replace the old string with the new
    
    # write the result into the destination file
    

In [ ]:
# Solution goes here

Per testare la tua funzione, leggi il file `photos/notes.txt`, sostituisci `'photos'` con `'images'` e scrivi il risultato nel file `photos/new_notes.txt`.

In [ ]:
source_path = 'photos/notes.txt'
open(source_path).read()

In [ ]:
dest_path = 'photos/new_notes.txt'
old = 'photos'
new = 'images'
replace_all(old, new, source_path, dest_path)

In [ ]:
open(dest_path).read()

### Esercizio

In [una sezione precedente](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap13.ipynb) abbiamo usato il modulo `shelve` per creare un archivio chiave-valore che fa corrispondere a una stringa di lettere ordinate una lista di anagrammi.
Per completare l'esempio, scrivi una funzione chiamata `add_word` che riceve come argomenti una stringa e un oggetto shelf.

Deve ordinare le lettere della parola per creare una chiave, poi verificare se la chiave è già nello shelf. Se non c'è, deve creare una lista che contiene la nuova parola e aggiungerla allo shelf. Se c'è, deve aggiungere la nuova parola al valore esistente.

In [ ]:
# Solution goes here

Puoi usare questo ciclo per testare la tua funzione.

In [ ]:
download('https://raw.githubusercontent.com/AllenDowney/ThinkPython/v3/words.txt');

In [ ]:
word_list = open('words.txt').read().split()

db = shelve.open('anagram_map', 'n')
for word in word_list:
    add_word(word, db)

Se tutto funziona, dovresti riuscire a cercare una chiave come `'opst'` e ottenere una lista di parole che si possono comporre con quelle lettere.

In [ ]:
db['opst']

In [ ]:
for key, value in db.items():
    if len(value) > 8:
        print(value)

In [ ]:
db.close()

### Esercizio

In una grande raccolta di file, potrebbe esserci più di una copia dello stesso file, memorizzata in directory diverse o con nomi di file diversi.
L'obiettivo di questo esercizio è cercare i duplicati.
Come esempio, lavoreremo con i file di immagine nella directory `photos`.

Ecco come funzionerà:

* Useremo la funzione `walk` della sezione [Esplorare le directory](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap13.ipynb) per cercare in questa directory i file che terminano con una delle estensioni in `config['extensions']`.

* Per ogni file, useremo `md5_digest` della sezione [Verificare file equivalenti](https://colab.research.google.com/github/pensare-in-python/3a-edizione/blob/main/notebooks/chap13.ipynb) per calcolare un digest del contenuto.

* Usando uno shelf, creeremo una corrispondenza da ogni digest a una lista dei percorsi che hanno quel digest.

* Infine, cercheremo nello shelf i digest che corrispondono a più file.

* Se ne troviamo, useremo `same_contents` per confermare che i file contengono gli stessi dati.

Ti suggerirò alcune funzioni da scrivere prima, poi metteremo tutto insieme.

1. Per identificare i file di immagine, scrivi una funzione chiamata `is_image` che riceve un percorso e una lista di estensioni di file, e restituisce `True` se il percorso termina con una delle estensioni della lista. Suggerimento: usa `os.path.splitext` – oppure chiedi a un assistente virtuale di scrivere questa funzione per te.

In [ ]:
# Solution goes here

Puoi usare `doctest` per testare la tua funzione.

In [ ]:
from doctest import run_docstring_examples

def run_doctests(func):
    run_docstring_examples(func, globals(), name=func.__name__)

run_doctests(is_image)

2. Scrivi una funzione chiamata `add_path` che riceve come argomenti un percorso e uno shelf. Deve usare `md5_digest` per calcolare un digest del contenuto del file. Poi deve aggiornare lo shelf, creando un nuovo elemento che fa corrispondere al digest una lista contenente il percorso, oppure aggiungendo il percorso alla lista se esiste già.

In [ ]:
# Solution goes here

3. Scrivi una versione di `walk` chiamata `walk_images` che riceve una directory ed esplora i file della directory e delle sue sottodirectory. Per ogni file, deve usare `is_image` per verificare se è un file di immagine e `add_path` per aggiungerlo allo shelf.

In [ ]:
# Solution goes here

Quando tutto funziona, puoi usare il programma seguente per creare lo shelf, esplorare la directory `photos` aggiungendo i percorsi allo shelf, e poi verificare se ci sono più file con lo stesso digest.

In [ ]:
db = shelve.open('photos/digests', 'n')
walk_images('photos')

for digest, paths in db.items():
    if len(paths) > 1:
        print(paths)

Dovresti trovare una coppia di file che hanno lo stesso digest.
Usa `same_contents` per verificare se contengono gli stessi dati.

In [ ]:
# Solution goes here

[Pensare in Python](https://pensare-in-python.github.io/3a-edizione/) – traduzione italiana non ufficiale di [Think Python: 3rd Edition](https://allendowney.github.io/ThinkPython/index.html)

Copyright 2024 [Allen B. Downey](https://allendowney.com)

Licenza del codice: [MIT License](https://mit-license.org/)

Licenza del testo (originale e traduzione): [Creative Commons Attribuzione - Non commerciale - Condividi allo stesso modo 4.0 Internazionale](https://creativecommons.org/licenses/by-nc-sa/4.0/deed.it)